In [1]:
import argparse
import logging
import time

from helper import log_time
from scripts import embedder, lisa, steering, sfam
from scripts.debugging import (
    debug_generation,
    debug_repeating_outputs,
    debug_similar_sentences,
    debug_skipped_sentence_outputs,
)
from scripts.evaluate import evaluate_all
from scripts.sentence_generation import (
    compute_clusters,
    create_data_style_vectors,
    create_similarity_style_vectors,
    generate_style_descriptions,
    generate_style_sentences,
    select_clusters,
    split_style_sentences,
)
from utils.logger import setup_logging
from config import GENERATION_MODEL, LOGGING_FOLDER, SELECTED_KNOWLEDGE_PROMPT_IDS, SELECTED_STYLE_PROMPT_IDS
from utils.llama import Llama
from utils.database import Database

import pandas as pd
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import scipy.stats as stats
from collections import defaultdict
import logging
import re
import time
from typing import List, Literal
import numpy as np
import pandas as pd
from tqdm import tqdm

from config import (
    DESCRIPTION_GENERATION_ITERATIONS,
    MAX_GROUP_RATIO,
    MAX_NUM_CLUSTERS,
    MAX_SIMILARITY,
    MIN_GROUP_ABSOLUTE,
    MIN_GROUP_RATIO,
    MIN_KNOWLEDGE_ATTRIBUTES,
    MIN_SIMILARITY,
    PROCESS_CHUNK_SIZE,
    STYLE_VECTOR_SIZE,
)
from data import prompts
from data.prompts import get_sentence_prompt, get_knowledge_sentence_prompt
from helper import log_time
from utils.sentence_transformer import SentenceTransformerModel
from utils.llama import Llama
from utils.database import Database
from sklearn.feature_extraction.text import TfidfTransformer
from scipy.stats import pointbiserialr

In [2]:
DATABASE_NAME = "data.sqlite"
database = Database(DATABASE_NAME)
database.create_database()

In [9]:
prompt_type = ["Open prompt"]
prompt_type_query = ""
if len(prompt_type) > 0:
    prompt_type_query = "WHERE " + "(" + " OR ".join(
        [f"prompts.type = '{prompt}'" for prompt in prompt_type]
    ) + ")"

    all_prompt_ids = SELECTED_STYLE_PROMPT_IDS + SELECTED_KNOWLEDGE_PROMPT_IDS
    id_list_str = ", ".join(str(pid) for pid in all_prompt_ids)
    prompt_id_query = f"AND prompts.id IN ({id_list_str})"
else:
    all_prompt_ids = SELECTED_STYLE_PROMPT_IDS + SELECTED_KNOWLEDGE_PROMPT_IDS
    id_list_str = ", ".join(str(pid) for pid in all_prompt_ids)
    prompt_id_query = f"WHERE prompts.id IN ({id_list_str})"
    
df = pd.read_sql_query(
            f"""
            SELECT questions.data_group, clusters.id, clusters.center_id, style_sentences.style_sentence, COUNT(DISTINCT style_descriptions.id) as count
            FROM clusters
            JOIN style_sentences ON clusters.center_id = style_sentences.id
            JOIN join_sentence_description ON style_sentences.id = join_sentence_description.sentence_id
            JOIN style_descriptions ON join_sentence_description.description_id = style_descriptions.id
            JOIN answers ON answers.id = style_descriptions.answer_id
            JOIN questions ON questions.id = answers.question_id
            JOIN prompts ON style_descriptions.prompt_id = prompts.id
            {prompt_type_query}
            {prompt_id_query}
            AND clusters.status >= 2
            GROUP BY questions.data_group, clusters.center_id
            ORDER BY count DESC
            """,
            database._conn,
        )

In [10]:
df

,data_group,id,center_id,style_sentence,count
0,philosophy,13,80889,The author uses run-on sentences.,37
1,aviation,74,6008,The author is familiar with aviation terminology.,33
2,history,13,80889,The author uses run-on sentences.,30
3,aviation,13,80889,The author uses run-on sentences.,25
4,politics,13,80889,The author uses run-on sentences.,25
...,...,...,...,...,...
846,softwareengineering,7832,206946,The author provides detailed statistics.,1
847,softwareengineering,9496,207832,The author is familiar with online platforms.,1
848,softwareengineering,6416,208090,The author emphasizes the importance of learni...,1
849,softwareengineering,41123,208240,The author is a skilled analyst.,1


In [11]:
# Create a pivot table: rows = data_group, columns = center_id, values = count
# center_id is the id of the style_sentence
pivot = df.pivot_table(index='data_group', columns='style_sentence', values='count', fill_value=0)
pivot

style_sentence,The author acknowledges external sources.,The author acknowledges prior knowledge.,The author acknowledges the complexity of historical events.,The author acknowledges the lack of concrete evidence.,The author adds a touch of personality to the passage.,The author adds complexity to the argument.,The author advises against certain actions.,The author advises against over-engineering.,The author advocates for a particular course of action.,The author aims to make information accessible to a general audience.,...,The author's writing can be dense and difficult to follow.,The author's writing can be redundant.,The author's writing is austere.,The author's writing is dry.,The author's writing style is anecdotal.,The author's writing style is characterized by brevity.,The author's writing style is characterized by objectivity.,The author's writing style is experimental.,The author's writing style is mathematical.,The author's writing style is somewhat hesitant or tentative.
data_group,,,,,,,,,,,,,,,,,,,,,
aviation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
biology,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
chemistry,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0
cs,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,2.0,0.0,0.0,1.0,0.0,1.0,0.0
electronics,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0
gamedev,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
history,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,1.0
philosophy,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
physics,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
groups = pivot.index
unique_groups = np.unique(groups)
most_important_dims = {}

# Loop over each group
for group in unique_groups:
    # Create a binary indicator: 1 if the sample belongs to the current group, 0 otherwise.
    binary_labels = (groups == group).astype(int)

    # Initialize an array to store correlation coefficients for each dimension
    corr_coeffs = np.zeros(pivot.shape[1])

    # Compute the point-biserial correlation for each dimension
    for i, col in enumerate(pivot.columns):
        corr, _ = pointbiserialr(pivot[col].values, binary_labels)
        corr_coeffs[i] = corr  # correlation for dimension i
    
    top_dims = np.argsort(corr_coeffs)[::-1]
    top_corrs = corr_coeffs[top_dims]

    most_important_dims[group] = {"dimensions": top_dims, "correlations": top_corrs}


In [16]:
style_sentences = np.array(pivot.columns)#
all_style_sentences = []
for group in unique_groups:
    print(group)
    for dim, corr in zip(most_important_dims[group]["dimensions"][:100], most_important_dims[group]["correlations"][:100]):
        if corr > 0.95:
            print(style_sentences[dim])
        all_style_sentences.append(dim)
            #print(corr)
    print()
print(len(all_style_sentences), len(set(all_style_sentences)))

aviation
The author writes in an everyday style.
The author conveys critical information.
The author focuses on logistics.
The author emphasizes commitment.
The author uses non-technical explanations.
The author uses parentheses to add credibility to their argument.
The author uses dollar signs to denote mathematical expressions.
The author uses concrete examples to illustrate complex ideas.
The author uses "ac" as an abbreviation.
The author aims to make information accessible to a general audience.
The author mirrors the chaotic nature of the subject matter.
The author is knowledgeable about aerodynamics.
The author is familiar with aviation terminology.

biology
The author often breaks away from traditional grammatical rules.
The author presents complex information in a nuanced manner.
The author uses the word "pathogens".
The author uses "plausible".
The author is prone to digression and meandering.
The author structures arguments logically.
The author is familiar with medical term